# 05 — Incremental updates, late arrivals, and replay

**Learner scaffold · RetailFlow dataset 1.0.0**

Course lessons: Day 2, Lessons 9–10.

Goal: Insert 20 sales, correct one, ignore two replays, and prove repeat-safe results.

Before running: Run 03 and 04. The notebook also accepts an already-completed final state for a repeat run.

Writes: <prefix>_bronze_incremental and updates to <prefix>_silver_sales. Gold is rebuilt separately in 06.

Follow the playlist module order. Build Buddy will help you create the missing numbered exercise cells. The instructor has a separate completed reference. Step IDs are stable anchors for that authoring work. Saved outputs are blank unless explicitly verified locally; a passed local check does not imply Databricks execution.


## Your notebook grows through Build Buddy

Configuration and instructor utility cells are supplied so you can focus on the learning task. The numbered exercise cells have no solutions. Read the playlist instructions, attempt each cell, run it, explain its output, then ask for a hint if needed. A blank exercise cell is incomplete work. Do not use Run All expecting this scaffold to be a finished pipeline. The course revisits notebooks: Lesson 5 runs 02-02, 02-03 and 02-05; Lesson 6 then runs 02-04.


## Configuration

Use the same assigned names in every notebook. Each notebook defines its own variables and helpers; it does not depend on another notebook’s Python memory. Persistent tables are the handoff between notebooks.


In [0]:
# Edit these four values in EVERY notebook before running it.
CATALOG = "workspace"
SCHEMA = "default"
VOLUME = "retailflow_files"
TABLE_PREFIX = "rf_retailflow"

import re
for name in [CATALOG, SCHEMA, VOLUME, TABLE_PREFIX]:
    assert re.fullmatch(r"[a-z][a-z0-9_]*", name), "Use lowercase letters, digits, and underscores; start with a letter."
assert not any(name.startswith("your_") for name in [CATALOG, SCHEMA, VOLUME]), "Replace the example names with your assigned names."
assert TABLE_PREFIX != "rf_yourname", "Choose the table prefix assigned to you."
DATASET_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/retailflow-v1.0.0"
print("Dataset:", DATASET_ROOT)
print("Table prefix:", f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_")


Dataset: /Volumes/workspace/default/retailflow_files/retailflow-v1.0.0
Table prefix: workspace.default.rf_retailflow_


## Supplied support cells

These utilities keep repeated setup and checks in one place inside this notebook. The Python cleaner deliberately processes this tiny fixture on the driver so beginners can inspect each row. Spark handles DataFrames, joins, summaries, and Delta tables. This is a teaching pattern for 80 sales, not a large-scale ingestion design. The helper definitions below are part of the notebook and can be inspected. Learners will construct selected parts through Build Buddy in the later authoring phase.


In [0]:
from pathlib import Path
from decimal import Decimal, ROUND_HALF_UP, InvalidOperation
from datetime import date, datetime
from collections import Counter, defaultdict
import csv, json, re

def read_rows(relative_path):
    with (Path(DATASET_ROOT) / relative_path).open(encoding='utf-8', newline='') as stream:
        return list(csv.DictReader(stream))

def money(value):
    return Decimal(value).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)

def business_payload(row):
    return {key: value for key, value in row.items() if key != 'source_row_id'}

def business_records(rows):
    return sorted([business_payload(row) for row in rows], key=lambda row: row['sale_id'])

def dimension_maps():
    return (
        {r['store_id']: r for r in read_rows('data/stores.csv')},
        {r['product_id']: r for r in read_rows('data/products.csv')},
        {r['customer_id']: r for r in read_rows('data/customers.csv')},
    )

def clean_sales(raw_rows):
    """Readable Python cleaning for this tiny fixture; no distributed UDF required."""
    stores, products, customers = dimension_maps()
    candidates, rejected, repaired = [], [], []
    for raw in sorted(raw_rows, key=lambda row: row['source_row_id']):
        row = {key: value.strip() for key, value in raw.items()}
        for key in ['store_id', 'product_id', 'customer_id', 'payment_method']:
            row[key] = row[key].upper()
        if row['customer_id'] == 'UNKNOWN':
            row['customer_id'] = ''
        reason = None
        if any(not row[k] for k in ['sale_id', 'source_row_id', 'sale_date', 'store_id', 'product_id']):
            reason = 'REQUIRED_MISSING'
        if not reason:
            try:
                fmt = '%d/%m/%Y' if '/' in row['sale_date'] else '%Y-%m-%d'
                pattern = r'\d{2}/\d{2}/\d{4}' if '/' in row['sale_date'] else r'\d{4}-\d{2}-\d{2}'
                if not re.fullmatch(pattern, row['sale_date']):
                    raise ValueError('Date format mismatch')
                row['sale_date'] = datetime.strptime(row['sale_date'], fmt).date().isoformat()
            except ValueError:
                reason = 'DATE_INVALID'
        if not reason:
            if row['product_id'] not in products:
                reason = 'PRODUCT_UNKNOWN'
            elif row['store_id'] not in stores:
                reason = 'STORE_UNKNOWN'
            elif row['customer_id'] and row['customer_id'] not in customers:
                reason = 'CUSTOMER_UNKNOWN'
        if not reason:
            try:
                if not re.fullmatch(r'\d+', row['quantity']) or int(row['quantity']) < 1:
                    reason = 'QUANTITY_INVALID'
                elif not re.fullmatch(r'\d+', row['record_version']) or int(row['record_version']) < 1:
                    reason = 'VERSION_INVALID'
                else:
                    row['unit_price'] = format(money(row['unit_price'] or products[row['product_id']]['list_price']), '.2f')
                    row['discount_pct'] = row['discount_pct'] or '0'
                    price, discount = Decimal(row['unit_price']), Decimal(row['discount_pct'])
                    if not price.is_finite() or price <= 0:
                        reason = 'PRICE_INVALID'
                    elif not discount.is_finite() or not 0 <= discount <= 100:
                        reason = 'DISCOUNT_INVALID'
                    elif row['payment_method'] not in ['UPI', 'CARD', 'CASH']:
                        reason = 'PAYMENT_INVALID'
            except (ValueError, InvalidOperation):
                reason = 'NUMBER_INVALID'
        if reason:
            rejected.append(dict(raw, rejection_reason=reason))
        else:
            if row != raw:
                repaired.append(row['source_row_id'])
            candidates.append(row)
    groups = defaultdict(list)
    for row in candidates:
        groups[(row['sale_id'], row['record_version'])].append(row)
    accepted, duplicate_ids = [], []
    raw_by_id = {row['source_row_id']: row for row in raw_rows}
    for group in groups.values():
        if any(business_payload(row) != business_payload(group[0]) for row in group):
            rejected.extend(dict(raw_by_id[row['source_row_id']], rejection_reason='VERSION_CONFLICT') for row in group)
        else:
            accepted.append(group[0])
            duplicate_ids.extend(row['source_row_id'] for row in group[1:])
    return accepted, rejected, duplicate_ids, repaired

def current_versions(rows):
    state = {}
    for row in rows:
        old = state.get(row['sale_id'])
        if old is None or int(row['record_version']) > int(old['record_version']):
            state[row['sale_id']] = row
    return sorted(state.values(), key=lambda row: row['sale_id'])

def visible_returns(stage, sales_rows):
    rows = [r for r in read_rows('data/returns.csv') if stage == 'final' or r['available_batch'] == '01']
    sales = {row['sale_id']: row for row in sales_rows}
    assert len({r['return_id'] for r in rows}) == len(rows)
    returned = Counter()
    for row in rows:
        sale = sales[row['sale_id']]
        assert row['available_batch'] in ['01', '02']
        assert date.fromisoformat(row['return_date']) >= date.fromisoformat(sale['sale_date'])
        quantity = int(row['return_quantity'])
        assert quantity > 0
        returned[row['sale_id']] += quantity
        assert returned[row['sale_id']] <= int(sale['quantity'])
        assert Decimal(row['refund_amount']) == money(quantity * Decimal(sale['unit_price']) * (1 - Decimal(sale['discount_pct']) / 100))
    return rows


In [0]:
from pyspark.sql import functions as F, types as T

SALES_COLUMNS = ['source_row_id', 'sale_id', 'sale_date', 'store_id', 'product_id', 'customer_id', 'quantity', 'unit_price', 'discount_pct', 'payment_method', 'record_version']
SALES_SCHEMA = T.StructType([
    T.StructField(name, T.DateType() if name == 'sale_date' else
                  T.IntegerType() if name in ['quantity', 'record_version'] else
                  T.DecimalType(12, 2) if name == 'unit_price' else
                  T.DecimalType(5, 2) if name == 'discount_pct' else T.StringType(), True)
    for name in SALES_COLUMNS
])

def sales_frame(rows):
    typed = []
    for row in rows:
        values = []
        for name in SALES_COLUMNS:
            value = row[name]
            if name == 'sale_date': value = date.fromisoformat(value)
            elif name in ['quantity', 'record_version']: value = int(value)
            elif name in ['unit_price', 'discount_pct']: value = Decimal(value)
            elif name == 'customer_id' and value == '': value = None
            values.append(value)
        typed.append(tuple(values))
    return spark.createDataFrame(typed, SALES_SCHEMA)

def raw_frame(relative_path):
    return spark.read.option('header', True).option('inferSchema', False).option('mode', 'FAILFAST').csv(f'{DATASET_ROOT}/{relative_path}')

def table_name(suffix):
    assert re.fullmatch(r'[a-z][a-z0-9_]*', suffix)
    return f'{CATALOG}.{SCHEMA}.{TABLE_PREFIX}_{suffix}'

def assert_expected(frame, relative_path, ignore_provenance=False, frame_schema=None):
    """Compare a frame against checks/expected/<relative_path>.
    Spark Connect: analyze the schema once in the calling cell and pass frame_schema in."""
    if frame_schema is None:
        raise ValueError('Pass frame_schema=<frame>.schema from the calling cell.')
    columns = list(frame_schema.fieldNames())
    if ignore_provenance and 'source_row_id' in columns:
        columns.remove('source_row_id')
    actual = frame.select(*columns)
    # Read full files before dropping provenance so schema fields match CSV positions.
    expected = spark.read.csv(f'{DATASET_ROOT}/checks/expected/{relative_path}', schema=frame_schema, header=True, mode='FAILFAST').select(*columns)
    assert actual.exceptAll(expected).limit(1).count() == 0, f'Extra/different rows: {relative_path}'
    assert expected.exceptAll(actual).limit(1).count() == 0, f'Missing rows: {relative_path}'
    print('PASS:', relative_path)

def collected_sales(frame):
    rows = []
    for record in frame.select(*SALES_COLUMNS).collect():
        row = record.asDict()
        row['sale_date'] = row['sale_date'].isoformat()
        row['quantity'] = str(row['quantity'])
        row['record_version'] = str(row['record_version'])
        row['unit_price'] = format(row['unit_price'], '.2f')
        row['discount_pct'] = format(row['discount_pct'].normalize(), 'f')
        row['customer_id'] = row['customer_id'] or ''
        rows.append(row)
    return sorted(rows, key=lambda row: row['sale_id'])

def merge_counts(current, incoming):
    joined = incoming.alias('s').join(current.alias('t'), F.col('s.sale_id') == F.col('t.sale_id'), 'left')
    business_columns = [c for c in SALES_COLUMNS if c != 'source_row_id']
    equal = F.lit(True)
    for name in business_columns:
        equal = equal & F.col(f's.{name}').eqNullSafe(F.col(f't.{name}'))
    conflicts = joined.where((F.col('s.record_version') == F.col('t.record_version')) & ~equal)
    assert conflicts.limit(1).count() == 0, 'Equal-version business conflict: investigate before merge.'
    counts = {'insert': 0, 'update': 0, 'noop': 0}
    for row in joined.select(
        F.when(F.col('t.sale_id').isNull(), 'insert')
         .when(F.col('s.record_version') > F.col('t.record_version'), 'update')
         .otherwise('noop').alias('action')
    ).groupBy('action').count().collect():
        counts[row['action']] = row['count']
    return counts

def merge_into_silver(incoming):
    from delta.tables import DeltaTable
    assert incoming.groupBy('sale_id').count().where('count > 1').limit(1).count() == 0
    target = table_name('silver_sales')
    counts = merge_counts(spark.table(target), incoming)
    (DeltaTable.forName(spark, target).alias('t')
        .merge(incoming.alias('s'), 't.sale_id = s.sale_id')
        .whenMatchedUpdateAll(condition='s.record_version > t.record_version')
        .whenNotMatchedInsertAll().execute())
    return counts


## 05-01: Validate and deduplicate arrivals

**Core concept Vibe:** An arrival is not automatically a new business sale.

**Build Buddy action:** Validate batch 02 and select its highest valid version per sale ID.

**Expected checkpoint:** 23 validated rows: 20 new IDs, 2 existing identical versions, 1 higher-version correction.


In [0]:
# Build Buddy step 05-01
# An arrival is not automatically a new business sale: validate, deduplicate, pick highest version.
accepted02, rejected02, duplicate_ids02, repaired02 = clean_sales(read_rows('data/sales_batch_02.csv'))
arrivals = current_versions(accepted02)
print('Validated rows:              ', len(accepted02))
print('Rejected arrivals:            ', len(rejected02))
print('Duplicates within the batch:  ', len(duplicate_ids02))
print('Current versions per sale ID: ', len(arrivals))

silver_rows = collected_sales(spark.table(table_name('silver_sales')))
silver_by_id = {row['sale_id']: row for row in silver_rows}
new_ids = [row['sale_id'] for row in arrivals if row['sale_id'] not in silver_by_id]
corrections = [row['sale_id'] for row in arrivals
               if row['sale_id'] in silver_by_id and int(row['record_version']) > int(silver_by_id[row['sale_id']]['record_version'])]
replays = [row['sale_id'] for row in arrivals
           if row['sale_id'] in silver_by_id and int(row['record_version']) == int(silver_by_id[row['sale_id']]['record_version'])]
print('New sale IDs:              ', len(new_ids))
print('Identical replays:         ', replays)
print('Higher-version corrections:', corrections)
assert len(arrivals) == 23 and len(new_ids) == 20 and replays != [] and len(replays) == 2 and corrections == ['S0003']
print('CHECKPOINT 05-01 PASS: 23 validated rows = 20 new IDs + 2 identical replays + 1 correction (S0003)')


Validated rows:               23
Rejected arrivals:             0
Duplicates within the batch:   0
Current versions per sale ID:  23
New sale IDs:               20
Identical replays:          ['S0001', 'S0002']
Higher-version corrections: ['S0003']
CHECKPOINT 05-01 PASS: 23 validated rows = 20 new IDs + 2 identical replays + 1 correction (S0003)


## 05-02: Predict what the merge should do

**Core concept Vibe:** Insert, update, and no-op are different actions.

**Build Buddy action:** Inspect current Silver and calculate expected actions before writing.

**Expected checkpoint:** First run: 20 inserts, 1 update, 2 no-ops. A completed-state rerun: 23 no-ops.


In [0]:
# Build Buddy step 05-02
# Insert, update, and no-op are different actions: predict them before writing.
incoming = sales_frame(arrivals)
predicted = merge_counts(spark.table(table_name('silver_sales')), incoming)
print('Predicted actions:', predicted)
assert predicted == {'insert': 20, 'update': 1, 'noop': 2}
print('CHECKPOINT 05-02 PASS: 20 inserts, 1 update, 2 no-ops predicted before any write')


Predicted actions: {'insert': 20, 'update': 1, 'noop': 2}
CHECKPOINT 05-02 PASS: 20 inserts, 1 update, 2 no-ops predicted before any write


## 05-03: Apply higher versions only

**Core concept Vibe:** The latest business version replaces the previous current version.

**Build Buddy action:** Retain the raw incremental snapshot and run a conditional Delta MERGE.

**Expected checkpoint:** 80 current sales; S0003 quantity 4, version 2; S0080 remains on 2 January.


In [0]:
# Build Buddy step 05-03
# Retain the raw incremental snapshot in Bronze, then merge only higher versions into Silver.
raw_frame('data/sales_batch_02.csv').write.mode('overwrite').saveAsTable(table_name('bronze_incremental'))
applied = merge_into_silver(incoming)
print('Applied actions:', applied)
assert applied == predicted
final = spark.table(table_name('silver_sales'))
print('Silver rows after merge:', final.count())
assert final.count() == 80
assert_expected(final, 'final_sales.csv', ignore_provenance=True, frame_schema=final.schema)
print('CHECKPOINT 05-03 PASS: Bronze incremental persisted (23 raw rows); Silver now holds 80 current sales')


Applied actions: {'insert': 20, 'update': 1, 'noop': 2}
Silver rows after merge: 80
PASS: final_sales.csv
CHECKPOINT 05-03 PASS: Bronze incremental persisted (23 raw rows); Silver now holds 80 current sales


## 05-04: Replay the delivery

**Core concept Vibe:** Repeat-safe processing gives the same current state after a retry.

**Build Buddy action:** Capture all current fields, replay batch 02, and compare the complete state including provenance.

**Expected checkpoint:** Zero inserts, zero updates, 23 no-ops; every current field remains unchanged.


In [0]:
# Build Buddy step 05-04
# Repeat-safe processing: replaying the same delivery changes nothing.
before_rows = {row['sale_id']: row for row in collected_sales(spark.table(table_name('silver_sales')))}
replay = merge_into_silver(incoming)
print('Replay actions:', replay)
assert replay == {'insert': 0, 'update': 0, 'noop': 23}
after_rows = {row['sale_id']: row for row in collected_sales(spark.table(table_name('silver_sales')))}
assert before_rows == after_rows
print('CHECKPOINT 05-04 PASS: 23 no-ops on replay; the current state is identical')


Replay actions: {'insert': 0, 'update': 0, 'noop': 23}
CHECKPOINT 05-04 PASS: 23 no-ops on replay; the current state is identical


## 05-05: Replay the older file

**Core concept Vibe:** An old version must not undo a correction.

**Build Buddy action:** Replay the canonical initial batch and verify S0003 stays at version 2.

**Expected checkpoint:** 60 no-ops; final Silver remains identical. Late arrival and correction require old-date report updates.


In [0]:
# Build Buddy step 05-05
# An old version must not undo a correction: replay the canonical initial batch.
accepted01, rejected01, duplicate_ids01, repaired01 = clean_sales(read_rows('data/sales_batch_01.csv'))
old_replay = merge_into_silver(sales_frame(current_versions(accepted01)))
print('Older-file replay actions:', old_replay)
assert old_replay == {'insert': 0, 'update': 0, 'noop': 60}
current = {row['sale_id']: row for row in collected_sales(spark.table(table_name('silver_sales')))}
print('S0003 current version:', current['S0003']['record_version'])
assert current['S0003']['record_version'] == '2'
silver_final = spark.table(table_name('silver_sales'))
assert_expected(silver_final, 'final_sales.csv', ignore_provenance=True, frame_schema=silver_final.schema)
print('CHECKPOINT 05-05 PASS: 60 no-ops; S0003 stays at version 2; Silver still matches the final answer key')


Older-file replay actions: {'insert': 0, 'update': 0, 'noop': 60}
S0003 current version: 2
PASS: final_sales.csv
CHECKPOINT 05-05 PASS: 60 no-ops; S0003 stays at version 2; Silver still matches the final answer key


## Explain the timing

The file arrives on 6 January, but S0080 happened on 2 January. A filter that accepts only sale dates after 3 January would lose it. The correction to S0003 also changes 1 January. Notebook 06 rebuilds Gold from current Silver so both old dates are corrected. Bronze initial has 68 rows because this run used the dirty route; Bronze incremental has 23 rows. Do not confuse this physical history with the canonical clean route’s 83 arrivals or the 80 current business sales.
